# Colab 整局落子序列 Transformer 風格編碼器訓練與推論啟動器

對應 `aicup/train_seq_emb.py` 與 `aicup/infer_seq_emb.py`，checkpoint 與快取存到 Drive `MyDrive/aicup-2026/player-trained-models/`。

1. 執行階段選 GPU（建議 A100 或 L4）
2. 確認 Drive `MyDrive/aicup-2026/move-seq-features-v180000/` 內有 `train.npz`、`newdev.npz`、`val648.npz`
3. 修改第一格 `MODE`（`smoke_nod4` / `smoke_d4` / `full_nod4` / `full_d4` / `infer_val` / `infer_b` / `infer_a`）與 `BRANCH`；`infer_*` 需填 `INFER_CKPT`
4. 執行階段 → 全部執行

In [ ]:
MODE = 'smoke_nod4'  # 'smoke_nod4', 'smoke_d4', 'full_nod4', 'full_d4', 'infer_val', 'infer_hold', 'infer_b', 'infer_a'
BRANCH = 'main'
SEQ_DIR = 'move-seq-features-v180000'
INFER_CKPT = ''  # infer_* 模式填寫 best_model.pth，例如 'player-trained-models/seq-nod4-<hash>/best_model.pth'
INFER_TTA = 1  # 無 D4 組填 1，D4 訓練組填 8
FULL_EPOCHS = 35
EXTRA_ARGS = []
AUTO_UNASSIGN = True

VALID_MODES = ('smoke_nod4', 'smoke_d4', 'full_nod4', 'full_d4', 'infer_val', 'infer_hold', 'infer_b', 'infer_a')
if MODE not in VALID_MODES:
    raise SystemExit('MODE 必須為 %s' % (VALID_MODES,))
if MODE.startswith('infer_') and not INFER_CKPT:
    raise SystemExit('推論模式請先填寫 INFER_CKPT')

In [ ]:
import os, subprocess, sys
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
DRIVE_ROOT = '/content/drive/MyDrive/aicup-2026'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('fetch', '-q', 'origin')
git('checkout', '-q', BRANCH)
git('pull', '-q', 'origin', BRANCH)
print(git('log', '-1', '--oneline'))

# 安裝 sgfmill（test 模式解析 SGF 需要）
try:
    import sgfmill  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)

def link_dir(name, target, must_exist=False):
    if must_exist and not os.path.isdir(target):
        raise SystemExit('Drive 缺少 %s，請先從本機上傳' % target)
    os.makedirs(target, exist_ok=True)
    if os.path.islink(name):
        os.unlink(name)
    if not os.path.exists(name):
        os.symlink(target, name)
    print('%-28s -> %s' % (name, target))

link_dir(SEQ_DIR, os.path.join(DRIVE_ROOT, SEQ_DIR), must_exist=True)
link_dir('tests', os.path.join(DRIVE_ROOT, 'tests'))
link_dir('player-trained-models', os.path.join(DRIVE_ROOT, 'player-trained-models'))

!nproc
!nvidia-smi

In [ ]:
HASH = git('rev-parse', '--short', 'HEAD')


def run_cmd(cmd):
    print('執行:', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise SystemExit('指令失敗，exit code %d' % p.returncode)


if MODE in ('smoke_nod4', 'smoke_d4'):
    use_d4 = MODE == 'smoke_d4'
    tag = 'd4' if use_d4 else 'nod4'
    CKPT_DIR = 'player-trained-models/seq-smoke-%s-%s' % (tag, HASH)
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_seq_emb',
        '--seq-dir', SEQ_DIR,
        '--ckpt-dir', CKPT_DIR,
        '--epochs', '5',
        *(['--augment-d4'] if use_d4 else []),
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
    # 冒煙完成後立刻產出 newdev 快取供本機核對 Gate 1
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_seq_emb',
        '--mode', 'packed',
        '--ckpt', os.path.join(CKPT_DIR, 'best_model.pth'),
        '--seq-dir', SEQ_DIR,
        '--split', 'newdev',
        '--tta', '8' if use_d4 else '1',
        '--out', os.path.join(CKPT_DIR, 'emb-cache', 'newdev_emb.npz'),
    ])
elif MODE in ('full_nod4', 'full_d4'):
    use_d4 = MODE == 'full_d4'
    tag = 'd4' if use_d4 else 'nod4'
    CKPT_DIR = 'player-trained-models/seq-%s-%s' % (tag, HASH)
    cmd = [
        sys.executable, '-u', '-m', 'aicup.train_seq_emb',
        '--seq-dir', SEQ_DIR,
        '--ckpt-dir', CKPT_DIR,
        '--epochs', str(FULL_EPOCHS),
        *(['--augment-d4'] if use_d4 else []),
        *EXTRA_ARGS,
    ]
    run_cmd(cmd)
elif MODE == 'infer_val':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    for split in ('newdev', 'val648'):
        run_cmd([
            sys.executable, '-u', '-m', 'aicup.infer_seq_emb',
            '--mode', 'packed',
            '--ckpt', INFER_CKPT,
            '--seq-dir', SEQ_DIR,
            '--split', split,
            '--tta', str(INFER_TTA),
            '--out', os.path.join(CACHE_DIR, split + '_emb.npz'),
            *EXTRA_ARGS,
        ])
elif MODE == 'infer_hold':
    # hold.npz 由本機 pack_move_seq --extra-split hold=... --splits hold 產生
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_seq_emb',
        '--mode', 'packed',
        '--ckpt', INFER_CKPT,
        '--seq-npz', os.path.join(SEQ_DIR, 'hold.npz'),
        '--tta', str(INFER_TTA),
        '--out', os.path.join(CACHE_DIR, 'hold_emb.npz'),
        *EXTRA_ARGS,
    ])
elif MODE == 'infer_b':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_seq_emb',
        '--mode', 'test',
        '--ckpt', INFER_CKPT,
        '--groups', 'b',
        '--tta', str(INFER_TTA),
        '--out-dir', CACHE_DIR,
        *EXTRA_ARGS,
    ])
elif MODE == 'infer_a':
    CACHE_DIR = os.path.join(os.path.dirname(INFER_CKPT), 'emb-cache')
    run_cmd([
        sys.executable, '-u', '-m', 'aicup.infer_seq_emb',
        '--mode', 'test',
        '--ckpt', INFER_CKPT,
        '--groups', 'a',
        '--tta', str(INFER_TTA),
        '--out-dir', CACHE_DIR,
        *EXTRA_ARGS,
    ])

In [ ]:
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()